# 拣金：Colab 摸底

目的：测出 Robometer 和 TOPReward 在 Colab 上的真实接口、速度和显存，好写适配器。大约 15 到 30 分钟。

**怎么跑**
1. 菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**（有 Pro 就选 L4 或 A100）。
2. 左边「文件」面板，把本机的 `dist/jianjin-0.1.0-py3-none-any.whl` 拖进 `/content/`。
3. 菜单「代码执行程序 → 全部运行」。有一格报错没关系，后面照常跑，报错本身也是结论。
4. 跑完后「文件 → 下载 → 下载 .ipynb」，把文件放到项目的 `notebooks/colab_probe.out.ipynb`，告诉 Claude。

模型权重来自 Hugging Face 公开仓库（Robometer-4B 约 9 GB，Qwen3-VL-8B 约 16 GB），不需要登录。


In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2. 装依赖
!pip install -q "lerobot[topreward]" 2>&1 | tail -3
!pip install -q "lerobot[robometer]" 2>&1 | tail -3
!pip install -q /content/jianjin-0.1.0-py3-none-any.whl 2>&1 | tail -3
import importlib.metadata as md
for p in ["lerobot", "torch", "transformers", "jianjin", "av"]:
    try:
        print(p, md.version(p))
    except Exception as e:
        print(p, "没装上", e)

In [ ]:
# 3. 从 Zenodo 按需取 RoboVAD：标注、关节数据，加 3 条示范的左路视频
import csv, io, json, traceback, zipfile
PICK = []
try:
    from jianjin.eval.robovad_fetch import HttpRangeFile, ROBOVAD_URL, SMALL_PREFIXES, extract
    z = zipfile.ZipFile(io.BufferedReader(HttpRangeFile(ROBOVAD_URL), buffer_size=1 << 20))
    extract(z, "/content/robovad", SMALL_PREFIXES)
    ann = json.load(open("/content/robovad/annotations.json"))
    test = [r["episode_name"] for r in csv.DictReader(open("/content/robovad/splits/test.csv"))]
    normal = next(n for n in test if n.startswith("h_pickandplace_ep") and n not in ann)
    rings = next(n for n in test if n.startswith("h_rings_on_peg") and n in ann)
    PICK = ["h_pickandplace_ep9", normal, rings]
    extract(z, "/content/robovad", tuple(f"RoboVAD/videos/{n}_left.mp4" for n in PICK))
    for n in PICK:
        print(n, ann.get(n, "正常"))
except Exception:
    traceback.print_exc()

In [ ]:
# 4. 读帧：每秒 2 帧，长边 448
import time, traceback
CLIPS = {}
try:
    from jianjin.ingest.robovad import read_robovad
    from jianjin.video import read_clip
    eps = {e.id: e for e in read_robovad("/content/robovad", names=PICK).episodes}
    for n in PICK:
        t = time.time()
        frames, times = read_clip(eps[n].cameras["left"], sample_fps=2.0, max_side=448)
        CLIPS[n] = (frames, times, eps[n].task)
        print(n, frames.shape, f"读帧 {time.time() - t:.1f}s", eps[n].task)
except Exception:
    traceback.print_exc()

In [ ]:
# 5. Robometer：加载、看接口源码
import inspect, traceback, torch
try:
    from lerobot.rewards.robometer import RobometerConfig, RobometerRewardModel
    from lerobot.rewards.robometer.modeling_robometer import ROBOMETER_FEATURE_PREFIX
    from lerobot.rewards.robometer.processor_robometer import RobometerEncoderProcessorStep
    rcfg = RobometerConfig(pretrained_path="lerobot/Robometer-4B", device="cuda", reward_output="progress")
    print(rcfg)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    rm = RobometerRewardModel.from_pretrained(rcfg.pretrained_path, config=rcfg)
    print(f"加载 {time.time() - t:.0f}s，显存 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print("参数 dtype", next(rm.parameters()).dtype, "设备", next(rm.parameters()).device)
    if torch.cuda.get_device_capability()[0] < 8 and next(rm.parameters()).dtype == torch.bfloat16:
        rm = rm.to(torch.float16)
        print("这块卡不支持 BF16，转成 FP16")
    print("forward 签名", inspect.signature(rm.forward))
    print("==== compute_reward 源码 ====")
    print(inspect.getsource(rm.compute_reward)[:6000])
except Exception:
    traceback.print_exc()

In [ ]:
# 6. Robometer：整段视频调一次，看返回值、耗时、显存
import traceback


def to_cuda(batch):
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in batch.items()}


try:
    enc = RobometerEncoderProcessorStep(
        base_model_id=rcfg.base_model_id,
        use_multi_image=rcfg.use_multi_image,
        use_per_frame_progress_token=rcfg.use_per_frame_progress_token,
        max_frames=rcfg.max_frames,
    )
    for n in PICK:
        frames, times, task = CLIPS[n]
        encoded = enc.encode_samples([(frames, task)])
        batch = to_cuda({f"{ROBOMETER_FEATURE_PREFIX}{k}": v for k, v in encoded.items()})
        if n == PICK[0]:
            print("batch 里的键和形状", {k: getattr(v, "shape", type(v)) for k, v in batch.items()})
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        with torch.no_grad():
            r = rm.compute_reward(batch)
        print(n, "返回", type(r), getattr(r, "shape", None), r, f"{time.time() - t:.1f}s",
              f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
except Exception:
    traceback.print_exc()

In [ ]:
# 7. Robometer：模型内部有没有逐帧进度（看 forward 的输出和建模源码）
import traceback
try:
    with torch.no_grad():
        out = rm.forward(batch)
    print(type(out))
    if isinstance(out, dict):
        for k, v in out.items():
            print(k, getattr(v, "shape", type(v)))
    elif isinstance(out, (tuple, list)):
        for i, v in enumerate(out):
            print(i, getattr(v, "shape", type(v)))
    else:
        print(out)
except Exception:
    traceback.print_exc()
try:
    import lerobot.rewards.robometer.modeling_robometer as mr
    src = inspect.getsource(mr)
    i = src.find("progress")
    print("==== modeling_robometer.py 里提到 progress 的附近 ====")
    print(src[max(0, i - 2000): i + 4000])
except Exception:
    traceback.print_exc()

In [ ]:
# 8. Robometer：前 k 帧逐段调用 8 次，拼进度曲线，记总耗时
import numpy as np, traceback
try:
    for n in PICK:
        frames, times, task = CLIPS[n]
        ks = np.unique(np.linspace(2, len(frames), 8).astype(int))
        t = time.time()
        prog = []
        for k in ks:
            encoded = enc.encode_samples([(frames[:k], task)])
            b = to_cuda({f"{ROBOMETER_FEATURE_PREFIX}{kk}": v for kk, v in encoded.items()})
            with torch.no_grad():
                prog.append(float(torch.as_tensor(rm.compute_reward(b)).flatten()[0]))
        print(n, "锚点秒数", [round(float(times[k - 1]), 1) for k in ks],
              "进度", [round(p, 2) for p in prog], f"共 {time.time() - t:.1f}s")
except Exception:
    traceback.print_exc()

In [ ]:
# 9. 腾显存给 TOPReward
import gc
try:
    del rm
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"现在占用 {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# 10. TOPReward：先试 8B，装不下换 4B；看接口源码
import traceback
TOP = None
try:
    from lerobot.rewards import make_reward_model, make_reward_model_config, make_reward_pre_post_processors
    for vlm in ["Qwen/Qwen3-VL-8B-Instruct", "Qwen/Qwen3-VL-4B-Instruct"]:
        try:
            torch.cuda.reset_peak_memory_stats()
            t = time.time()
            tcfg = make_reward_model_config("topreward", vlm_name=vlm, device="cuda", image_key="observation.images.top")
            tm = make_reward_model(tcfg)
            tpre, tpost = make_reward_pre_post_processors(tcfg)
            TOP = vlm
            print(vlm, f"加载 {time.time() - t:.0f}s，显存 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
            break
        except Exception as e:
            print(vlm, "加载失败：", repr(e)[:500])
            gc.collect()
            torch.cuda.empty_cache()
    print(tcfg)
    print("==== compute_reward 源码 ====")
    print(inspect.getsource(tm.compute_reward)[:6000])
    print("==== 预处理器 ====")
    print(tpre)
except Exception:
    traceback.print_exc()

In [ ]:
# 11. TOPReward：按预处理器的格式喂一段视频，看返回值、耗时
import traceback
try:
    frames, times, task = CLIPS[PICK[0]]
    video = torch.from_numpy(frames).permute(0, 3, 1, 2).float().div(255).unsqueeze(0)  # 1×T×C×H×W
    raw = {"observation.images.top": video, "task": [task]}
    try:
        b = tpre(raw)
    except Exception as e:
        print("直接过预处理器失败，打印预处理器各步源码：", repr(e)[:500])
        for step in getattr(tpre, "steps", []):
            print("----", type(step).__name__)
            print(inspect.getsource(type(step))[:3000])
        raise
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    with torch.no_grad():
        r = tm.compute_reward(b)
    print("返回", type(r), getattr(r, "shape", None), r, f"{time.time() - t:.1f}s",
          f"峰值 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
except Exception:
    traceback.print_exc()

In [ ]:
# 12. 小结
print("GPU", torch.cuda.get_device_name(0), "；TOPReward 用的", TOP)
print("跑完了。菜单「文件 → 下载 → 下载 .ipynb」，放到项目 notebooks/colab_probe.out.ipynb。")